## Step 1 – Setup

In [ ]:
pip install pytrends gspread oauth2client openai pandas

## Step 2 – Fetch Trending Job Topics from Google Trends

In [ ]:
from pytrends.request import TrendReq

# Initialize pytrends
pytrends = TrendReq(hl='en-US', tz=330)

# Search queries related to jobs
keywords = ["admit card", "job notification", "exam result"]

trending_data = []
for kw in keywords:
    pytrends.build_payload([kw], cat=0, timeframe='now 7-d', geo='IN', gprop='')
    data = pytrends.related_queries()[kw]['top']
    if data is not None:
        trending_data.extend(data['query'].tolist())

# Remove duplicates
trending_data = list(set(trending_data))

print("Trending Job Topics:")
print(trending_data)


Trending Job Topics:
['ssc admit card', 'lic admit card', 'sarkari result 2025', 'sarkari result info', 'ssc', 'rrb notification', 'ibps clerk', 'free job alert', 'ibps clerk admit card 2025', 'ibps po exam result', 'ib', 'ibps po exam result 2025', 'ibps po prelims result', 'ignou', 'lic aao admit card', 'job notification 2025', 'exam result info', 'free job alert notification', 'ibps admit card', 'ignou exam result', 'government job notification', 'rrb ntpc', 'cgl admit card', 'ssc notification 2025', 'ssc admit card 2025', 'ibps clerk admit card', 'admit card download', 'sarkari job', 'ibps po result', 'ap job notification 2025', 'free job alert 2025', 'cisf admit card', 'ibps po exam date 2025', 'ssc cgl admit card', 'lic aao', 'lic aao admit card 2025', 'sarkari exam result', 'ibps po result 2025', 'cisf', 'free job notification 2025', 'rrb', 'job alert 2025', 'rrb job notification 2025', 'ib admit card', 'rrb notification 2025', 'exam sarkari result info', 'ignou result', 'govt j

## Step 3 – Save Data to Google Sheets

In [5]:
import gspread
from oauth2client.service_account import ServiceAccountCredentials

# --- Google Sheets Authentication ---
scope = ["https://spreadsheets.google.com/feeds",
         "https://www.googleapis.com/auth/drive"]

creds = ServiceAccountCredentials.from_json_keyfile_name("credentials.json", scope)
client = gspread.authorize(creds)

# --- Open your sheet ---
sheet = client.open("Job_Trends_Agent").sheet1

# --- Prepare header if empty ---
if not sheet.get_all_values():
    sheet.append_row(["Trend", "Category", "Instagram", "Blog", "Reel", "Thumbnail"])

# --- Get existing trends to avoid duplicates ---
existing = [row[0] for row in sheet.get_all_values()[1:]]  # skip header

# --- Collect new trends to insert ---
rows_to_insert = []
for trend in trending_data:
    if trend not in existing:
        rows_to_insert.append([trend, "", "", "", "", ""])

# --- Batch insert new rows in ONE API call ---
if rows_to_insert:
    sheet.append_rows(rows_to_insert, value_input_option='USER_ENTERED')
    print(f"{len(rows_to_insert)} new trends added to Google Sheet.")
else:
    print("No new trends to add.")


5 new trends added to Google Sheet.
